In [1]:
event = {
    "id": "https://example.com/event1",
    "label": "2e expeditie naar nias",
    "title": "2e expeditie naar nias",
    "timeSpan_start": "1863-01-01",
    "timeSpan_end": "1863-01-01",
    "date_y": 1863,
    "date_m": 1,
    "date_d": 1,
    "UTC_DATE": "1863-01-01T00:00:00Z",
    "fulltext": "",
    "entities": []
}

Given an event extract the articles

In [2]:
import xmltodict as xd
import json
import os
import requests
from utils import extract_str

def parse_resp_events(response, event):
    data = xd.parse(response.content, xml_attribs=False)
    raw_records = data['srw:searchRetrieveResponse']['srw:records']['srw:record']
    if not isinstance(raw_records, list):
        raw_records = [raw_records]
   
    # print(f"raw_records: {len(raw_records)} for response: {response.url}")

    # THIS IS TEMPORARY
    dir = os.getcwd()
    
    for record in raw_records:
        record_data = record['srw:recordData']
        # Load zones as json
        record_data['zones'] = json.loads(record_data['zones'])
        # Get OCR (text content of response)
        ocr_url = record_data['dc:identifier']
        with requests.get(ocr_url) as ocr_response:
            record_data['ocr'] = ocr_response.text
        record_data['event_id'] = event.get('id', '')
        record_data['event_title'] = event.get('title', '')

        identifier = "_".join(extract_str(record_data['dc:identifier'], '?urn=').split(':')[:-1])  # Extract identifier and remove 'ddd:' prefix
        filename = os.path.join(dir, 'DST', identifier + '.json')

        # print(f"{type(record_data)} for record_data type, {filename} for filename")

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(record_data, f, indent=2, ensure_ascii=False)

        

/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
base_EVENT_BASED_URL = f"https://jsru.kb.nl/sru/sru?version=1.2&operation=searchRetrieve&x-collection=DDD_artikel&recordSchema=indexing&startRecord=1&maximumRecords=1&query=(content all \"%s\") AND (date within \"%s %s\")"
# iter_EVENT_BASED_URL = f"https://jsru.kb.nl/sru/sru?version=1.2&operation=searchRetrieve&x-collection=DDD_artikel&recordSchema=indexing&startRecord=%s&maximumRecords=%s&query=\"%s\" AND (date within \"%s %s\")"
iter_EVENT_BASED_URL = f"https://jsru.kb.nl/sru/sru?version=1.2&operation=searchRetrieve&x-collection=DDD_artikel&startRecord=%s&maximumRecords=%s&query=(content all \"%s\") AND (date within \"%s %s\")&recordSchema=ddd&x-fields=zones"

import pandas as pd

import requests
import lxml.etree
def get_article_by_event(event:json):
    title = event.get("title", "")
    date_y = event.get("date_y", "")
    # print(f"Title: {title}, Fulltext: {fulltext}, Date: {date_y}")

    base_url = base_EVENT_BASED_URL % (title, date_y-10, date_y+10)

    resp = requests.get(base_url)
    data = lxml.etree.fromstring(resp.content)
    total_nr_results = 0
    for i in data.iter():
        if i.tag == '{http://www.loc.gov/zing/srw/}numberOfRecords':
            total_nr_results = int(i.text)
            break
    print(f"Total results: {total_nr_results} for event: {title}")

    if total_nr_results == 0:
        print(f"No results found for event: {title}")
        return
    
    inv = 10 if total_nr_results > 10 else total_nr_results

    result_dicts = []
    for start in range(1, total_nr_results+1, inv):
        paged_url = iter_EVENT_BASED_URL % (start, inv, title, date_y-10, date_y+10)
        # print(f"Fetching articles for event: {title} with paged URL: {paged_url}")
        paged_resp = requests.get(paged_url)
        parse_resp_events(paged_resp, event)

    # # convert list of dicts to dataframe
    # df = pd.DataFrame(result_dicts)
    # df.to_csv(f"{title.replace(' ', '_')}_articles.csv", index=False)
    

In [4]:
get_article_by_event(event)

Total results: 83 for event: 2e expeditie naar nias


Apply NER on each article

In [8]:
# We need the latest version of GLiNER, which is for some reason not available 
# through pypy (https://github.com/urchade/GLiNER/issues/360), we can install
# it from source.
os.chdir('../GLiNER')

import os

# !git clone https://github.com/urchade/GLiNER
# os.chdir('GLiNER')
!pip install -r requirements.txt -q
!pip install . -q


os.chdir('../src')
os.getcwd()

'/Users/sarah_shoilee/codeProjects/linking_delpher/src'

In [10]:
import torch
import transformers

print(torch.__version__)
print(transformers.__version__)
print(torch.cuda.is_available())

2.5.1
4.57.6
False


In [1]:
import gliner
assert gliner.__version__ == '0.2.27'
print(gliner.__version__)

0.2.27


In [2]:
from pathlib import Path

from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import nltk

# from data_utils import prepare_data, convert_to_dataset, ner_tags_to_spans

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sarah_shoilee/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
# Rename tags in datasets to have full tag names, which are used in the model

# English classes
ner_tag_to_class_mapping = {
    'O': 'O',
    'B-PER': 'B-person',
    'I-PER': 'I-person',
    'B-ORG': 'B-organization',
    'I-ORG': 'I-organization',
    'B-LOC': 'B-location',
    'I-LOC': 'I-location',
}

# Dutch classes
ner_tag_to_class_mapping = {
    'O': 'O',
    'B-PER': 'B-persoon',
    'I-PER': 'I-persoon',
    'B-ORG': 'B-organisatie',
    'I-ORG': 'I-organisatie',
    'B-LOC': 'B-locatie',
    'I-LOC': 'I-locatie',
}

# Generate label list
label_list = list(ner_tag_to_class_mapping.values())

# label_map = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}
tag_to_id = {label: i for i, label in enumerate(label_list)}
tag_to_id

{'O': 0,
 'B-persoon': 1,
 'I-persoon': 2,
 'B-organisatie': 3,
 'I-organisatie': 4,
 'B-locatie': 5,
 'I-locatie': 6}

In [4]:
from gliner import GLiNER
import gliner
import torch

# from custom_evaluation import evaluate, flatten_for_eval

In [5]:
model_id = "urchade/gliner_small"
model_id = "EmergentMethods/gliner_large_news-v2.1"
model_id = "urchade/gliner_multi-v2.1"


if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"


model = GLiNER.from_pretrained(
    model_id, 
    load_tokenizer=True, 
    map_location=device
)

/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# use_fine_grained = True
use_fine_grained = False

if not use_fine_grained:
    entity_types = list(set([i.split("-")[-1] for i in label_list if i != "O"]))
    # list(set([i.split("-")[-1] for i in ner_tag_to_class_mapping.values() if i != "O"]))
else:
    pass
    # entity_types = list(set(fine_grained_labels.keys()))

print(f"{len(entity_types)} entity types: {entity_types}")

3 entity types: ['organisatie', 'persoon', 'locatie']


In [7]:
text = "The quick brown fox jumps over the lazy dog. John Doe works at OpenAI in San Francisco."
model.predict_entities(text, entity_types)

[{'start': 45,
  'end': 53,
  'text': 'John Doe',
  'label': 'persoon',
  'score': 0.9795668721199036},
 {'start': 63,
  'end': 69,
  'text': 'OpenAI',
  'label': 'organisatie',
  'score': 0.9815333485603333},
 {'start': 73,
  'end': 86,
  'text': 'San Francisco',
  'label': 'locatie',
  'score': 0.9803043007850647}]

In [ ]:

def extract_names_entities(text:str):
    labels = ["person", "geographic location", "historical_event"]

    entities = model.predict_entities(text, labels)

    person_entities = [e for e in entities if e["label"] == "person"]
    location_entities = [e for e in entities if e["label"] == "geographic location"]
    organization_entities = [e for e in entities if e["label"] == "organization"]
    historical_event_entities = [e for e in entities if e["label"] == "historical_event"]

    return {
        "person": person_entities,
        "geographic location": location_entities,
        "organization": organization_entities,
        "historical_event": historical_event_entities
    }

TypeError: 

In [2]:
import os
os.getcwd()

'/Users/sarah_shoilee/codeProjects/linking_delpher/src'

In [ ]:
# travers through all file in a directory
DIR = os.path.join(os.getcwd(), 'DST')
for filename in os.listdir(DIR):
    if filename.endswith(".json"):
        filepath = os.path.join(DIR, filename)

    # read the json file and extract the ocr content
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    ocr_content = data.get('ocr', '')
    # read ocr content as XML and seperater text by paragraphs
    ocr_content = lxml.etree.fromstring(ocr_content.encode('utf-8')).xpath('//p/text()')

    data["person"] = []
    data["geographic location"] = []
    data["organization"] = []
    data["historical_event"] = []

    # print(f"Extracting entities for file: {filename} with OCR content length: {len(ocr_content)}")
    for i, paragraph in enumerate(ocr_content):
        for _, k in enumerate(extract_names_entities(paragraph)):
            print(f"Extracted {len(extract_names_entities(paragraph)[k])} entities for label: {k} in paragraph {i} of file: {filename}")    
            data[k] = data[k].extend(extract_names_entities(paragraph)[k])
    # apply GLiNER to extract entities

    print(f"Extracted entities for file: {filename} - {data['person']} persons, {data['geographic location']} locations, {data['organization']} organizations, {data['historical_event']} historical events")
    
    # add the extracted entities to the json file and save it again
    data['entities'] = extract_names_entities(ocr_content)  # This is a placeholder for the actual entity extraction function
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 1297 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


AttributeError: 'NoneType' object has no attribute 'extend'

Disambiguate person within article

Disambiguate person within event (across article)

Make list of person dictionary 